# Fixed-point branch initialization

This notebook is a visual, cell-by-cell interface to `fixed_point_init.py`. The script is the sole implementation and can run the same workflow end to end through its `main()` entry point.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from fixed_point_init import (
    SYSTEM,
    plot_sweep,
    plot_seed_trajectory,
    save_branch_seed,
    save_sweep,
    seed_branch,
    select_candidate,
    sweep_period_hessian,
)

## Configuration

The fixed phase-space state is expressed in radians and radians per unit time. Set `candidate_index` explicitly to reproduce a documented mode choice. Leaving it as `None` selects the global minimum of the restricted scan.

In [ ]:
initial_condition = (0.0, 0.0, 0.0, 0.0)
periods = np.linspace(0.01, 10.0, 1000)
frequency_cutoff = 16
stationary_starts = (True, True)
case_name = 'down_down'
candidate_index = None
branch_output = 'hess_results/bprop_up_down'

## Sweep the fixed-point Hessian

The combined plot uses a logarithmic eigenvalue axis to resolve near-zero minima. Nonpositive or nonfinite samples appear as gaps and are reported with their periods and values; they are not replaced by absolute values or a plotting floor. Inspect any warnings before selecting a seed.

In [ ]:
sweep = sweep_period_hessian(
    initial_condition,
    periods,
    frequency_cutoff=frequency_cutoff,
    stationary_starts=stationary_starts,
    compute_full_hessian=True,
)

In [ ]:
fig, ax = plot_sweep(sweep);

## Inspect and seed the selected mode

In [ ]:
selected_index = select_candidate(sweep, candidate_index)
{
    'index': selected_index,
    'period': sweep.periods[selected_index].item(),
    'minimum_eigenvalue': sweep.minimum_eigenvalues[selected_index].item(),
}

In [ ]:
seed = seed_branch(
    sweep,
    branch_output,
    candidate_index=selected_index,
    eigenvector_index=0,
    step_angle_degrees=0.5,
    orientation_component=0,
    orientation_sign=1,
)
{
    'initial_condition_degrees': (
        SYSTEM.z0(seed.branch.theta0).squeeze() * 180 / np.pi
    ).tolist(),
    'period': seed.candidate_period,
    'seed_loss': seed.seed_loss,
    'closure_error_degrees': seed.closure_error_degrees,
}

The seed is only a predictor and is not yet a converged periodic orbit. Compare its Fourier loop with direct integration in the angle-plane plot below. The start and integrated endpoint are marked; the title reports the full four-state closure error, since agreement in the angle projection alone does not establish closure. Save after confirming the selected mode and direction.

In [ ]:
fig, ax = plot_seed_trajectory(seed);

In [ ]:
save_sweep(sweep, 'fixed_point_evalues', case_name)
save_branch_seed(seed, overwrite=False)